# SPINE vis-style muon particle extraction

`spine.vis.out.Drawer`의 truth/reco point-key 매핑과 index 접근 방식을 그대로 따라
뮤온(track) 파티클을 **하나씩** 추출하는 노트북입니다.


In [ ]:
import numpy as np
import plotly.graph_objects as go

from spine.vis.out import Drawer
from spine.utils.globals import MUON_PID, TRACK_SHP


In [ ]:
# input path placeholder (사용자가 교체)
data_path = "directory/name"

# 실제 사용 시 아래에 로더 연결
# data = your_loader(data_path)
data = None


In [ ]:
def _drawer_truth_point_mapping(truth_point_mode):
    # spine.vis.out.Drawer._point_modes와 동일
    mapping = {
        "points": "points_label",
        "points_adapt": "points",
        "points_g4": "points_g4",
    }
    if truth_point_mode not in mapping:
        raise ValueError(f"Invalid truth_point_mode: {truth_point_mode}")
    return mapping[truth_point_mode]


def iter_muon_particles_vis_style(
    data,
    mode="truth",
    truth_point_mode="points",
    min_ke_mev=150.0,
    max_ke_mev=2000.0,
    require_contained=True,
    min_num_points=20,
):
    """Yield muon-track particles one by one, following Drawer point/index style."""
    particles_key = f"{mode}_particles"
    if particles_key not in data:
        raise KeyError(f"Missing `{particles_key}`")

    if mode == "truth":
        point_key = _drawer_truth_point_mapping(truth_point_mode)
        index_attr = truth_point_mode.replace("points", "index")
    elif mode == "reco":
        point_key = "points"
        index_attr = "index"
    else:
        raise ValueError("mode must be 'truth' or 'reco'")

    if point_key not in data:
        raise KeyError(f"Missing `{point_key}` for mode={mode}")

    points_tensor = data[point_key]

    for p in data[particles_key]:
        if p.shape != TRACK_SHP:
            continue
        if p.pid != MUON_PID:
            continue

        ke = getattr(p, "ke", -1.0)
        if not (min_ke_mev <= ke <= max_ke_mev):
            continue

        if require_contained and not getattr(p, "is_contained", False):
            continue

        idx = getattr(p, index_attr, None)
        if idx is None or len(idx) < min_num_points:
            continue

        yield {
            "particle": p,
            "particle_id": int(p.id),
            "points": np.asarray(points_tensor[idx], dtype=np.float32),
            "target_ke": float(ke),
            "num_points": int(len(idx)),
            "is_contained": bool(getattr(p, "is_contained", False)),
            "length": float(getattr(p, "length", -1.0)),
            "mode": mode,
            "truth_point_mode": truth_point_mode if mode == "truth" else None,
            "point_key": point_key,
            "index_attr": index_attr,
        }


In [ ]:
# one-by-one extraction
muon_samples = list(
    iter_muon_particles_vis_style(
        data,
        mode="truth",
        truth_point_mode="points",
        min_ke_mev=150.0,
        max_ke_mev=2000.0,
        require_contained=True,
        min_num_points=20,
    )
)

print(f"Extracted muon particles: {len(muon_samples)}")
if muon_samples:
    print({k: v for k, v in muon_samples[0].items() if k not in ['particle', 'points']})


In [ ]:
# visualize first extracted particle points
if muon_samples:
    s = muon_samples[0]
    pts = s["points"]

    fig = go.Figure([
        go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode="markers",
            marker=dict(size=2),
            name=f"particle_id={s['particle_id']} | KE={s['target_ke']:.1f} MeV",
        )
    ])
    fig.update_layout(title="First extracted muon particle")
    fig.show()


In [ ]:
# Drawer cross-check: exactly one selected particle draw
if muon_samples:
    s = muon_samples[0]
    pid0 = s["particle_id"]

    draw_data = dict(data)
    draw_data["truth_particles"] = [p for p in data["truth_particles"] if p.id == pid0]

    drawer = Drawer(draw_data, draw_mode="truth", truth_point_mode="points")
    fig = drawer.get(
        "particles",
        attr=["pid", "shape", "id", "length", "ke"],
        color_attr="pid",
        draw_end_points=True,
        draw_directions=True,
        split_traces=True,
    )
    fig.show()
